In [1]:
import subprocess
import re
import os

# analyzeVideo takes video and if nessesery creates new video file 
# returns information what fields were adjusted as string 
#  - analyze video (get info for every field)
#  - compare values with specified values 
#  - if nessesery make new file with changes 
#  - return text which field were adjusted 
def analyzeVideo(fileName, newFilePath):
  # s for specified 
  sVideoFormat = 'mp4'
  sVideoCodec = 'h264'
  sWidth = '640'
  sHeight = '360'
  sFrameRate = '25/1'
  sAspectRatio = '16:9'
  sVideoBitRateMin = '2097152' # 2 Mb is 2097152 bytes
  sVideoBitRateMax = '5242880' # 5 Mb is 5242880 bytes 
  sAudioCodec = 'aac'
  sAudioBitRateMax = '262144' # 256 kb is 262144 bytes 
  sAudioChannels = 'stereo'

  # get video info using ffmpeg command 
  videoRawOutput = subprocess.getoutput('ffprobe -v error -select_streams v -show_entries stream=display_aspect_ratio,codec_name,r_frame_rate,width,height,bit_rate -of default=noprint_wrappers=1 ' + fileName)
  audioRawOutput = subprocess.getoutput('ffprobe -v error -select_streams a -show_entries stream=codec_name,bit_rate,channel_layout -of default=noprint_wrappers=1 ' + fileName)

  # output above looks like that: 'codec_name=h264\nwidth=628\nheight=354\ndisplay_aspect_ratio=314:177\nr_frame_rate=30000/1001\nbit_rate=2989377'
  # use regex to get each information for each field 
  videoFormat = fileName[-3:]
  videoCodec = re.search(r'(?<=codec_name=)[^\n]+', videoRawOutput).group(0)
  width = re.search(r'(?<=width=)[^\n]+', videoRawOutput).group(0)
  height = re.search(r'(?<=height=)[^\n]+', videoRawOutput).group(0)
  frameRate = re.search(r'(?<=r_frame_rate=)[^\n]+', videoRawOutput).group(0)
  aspectRatio = re.search(r'(?<=display_aspect_ratio=)[^\n]+', videoRawOutput).group(0)
  videoBitRate = re.search(r'(?<=bit_rate=)[^\n]+', videoRawOutput).group(0)
  audioCodec = re.search(r'(?<=codec_name=)[^\n]+', audioRawOutput).group(0)
  audioBitRate = re.search(r'(?<=bit_rate=)[^\n]+', audioRawOutput).group(0)
  audioChannels = re.search(r'(?<=channel_layout=)[^\n]+', audioRawOutput).group(0)

  # get command string ready 
  command = 'ffmpeg -i ' + fileName
  justFileName = os.path.splitext(os.path.basename(fileName))[0]
  report = justFileName
  

  # -- Compare specified with current video --
  # if different 
  #   add to chain ffmpeg command
  #   add to report what field is it 
  if videoFormat != sVideoFormat:
    report += '\n- ' + videoFormat + ' video format should be ' + sVideoFormat

  if videoCodec != sVideoCodec:
    command += ' -c:v libx264'
    report += '\n- ' + videoCodec + ' video codec should be ' + sVideoCodec

  if frameRate != sFrameRate:
    command += ' -r 25'
    report += '\n- ' + frameRate + ' frame rate should be ' + sFrameRate

  if width != sWidth or height != sHeight:
    command += ' -vf scale={0}:{1}'.format(sWidth, sHeight)
    if width != sWidth and height != sHeight:
      report += '\n- ' + width + ' width should be ' + sWidth
      report += '\n- ' + height + ' height should be ' + sHeight
    elif width != sWidth:
      report += '\n- ' + width + ' width should be ' + sWidth
    else:
      report += '\n- ' + height + ' height should be ' + sHeight
      
  if aspectRatio != sAspectRatio:
    command += ' -aspect {0}'.format(sAspectRatio)
    report += '\n- ' + aspectRatio + ' aspect ratio should be ' + sAspectRatio

  if videoBitRate < sVideoBitRateMin:
    command += ' -b:v 2.5M'
    report += '\n- ' + videoBitRate + ' video bit rate should at least ' + sVideoBitRateMin

  if videoBitRate > sVideoBitRateMax:
    command += ' -b:v 2.5M'
    report += '\n- ' + videoBitRate + ' video bit rate should be no more than ' + sVideoBitRateMax

  if audioCodec != sAudioCodec:
    command += ' -c:a aac'
    report += '\n- ' + audioCodec + ' audio codec should be ' + sAudioCodec

  if audioBitRate > sAudioBitRateMax:
    command += ' -b:a 0.25M'
    report += '\n- ' + audioBitRate + ' audio bit rate should be less than ' + sAudioBitRateMax

  if audioChannels != sAudioChannels:
    command += ' -ac 2'
    report += '\n- ' + audioChannels + ' audio chanels should be ' + sAudioChannels

  # does this file need any adjustments 
  if report != justFileName:
    # file name without path or extension 
    command += ' ' + newFilePath + '/' + justFileName + '_formatOK.mp4'
    subprocess.getoutput(command)
    # return report string with information which fields had to be adjusted 
    return report
  # nothing had to be adjusted return empty string 
  return ''



# function takes folder creates text file with report and processes every file in that folder 
def processFolder(folder, folderForNewFiles):
  # create report string 
  report = '\t-  Report -\nFollowing files had to be adjusted:'

  # loop over files that have to be processed 
  for file in os.listdir(folder):
    if file == '.DS_Store': # on my mac there is always.DS_Store file skip it 
        continue 
    filePath = os.path.join(folder, file)
    # process each file analyzeVideo will return string with information what had to be adjusted add it to report 
    report += '\n\n' + analyzeVideo(filePath, folderForNewFiles)

  # save that report into the txt file 
  with open(folderForNewFiles + "/report.txt", 'w') as file:
    file.write(report)
  return 


In [2]:
# make sure Formated folder is empty to run again
processFolder('Exercise3_Films', 'Formated')